# Mini-Batch Gradient Descent

**Date:** 2026-04-26

---

## What is Mini-Batch Gradient Descent?

**Mini-Batch GD** splits the dataset into small batches of size **B** (typically 32, 64, 128, 256) and performs one parameter update per batch.

It sits exactly between Batch GD and SGD:
- **B = 1** → degenerates to SGD
- **B = m** → degenerates to Batch GD
- **B = 32~256** → Mini-Batch GD ← **this is what everyone uses**

### Update Rule

$$\theta = \theta - \alpha \cdot \frac{1}{B} \sum_{i \in \text{batch}} \nabla_{\theta} L(x_i, y_i)$$

- $B$ — batch size
- $\lfloor m/B \rfloor$ updates happen per epoch

### Pros
- **GPU-friendly** — vectorized operations over a batch are highly parallelizable
- **Less noisy than SGD** — averaging over B samples smooths the gradient
- **Faster than Batch GD** — updates happen $m/B$ times per epoch, not once
- **Most practical** — default in PyTorch, TensorFlow, Keras

### Cons
- One extra hyperparameter: batch size B
- Gradient is still an approximation (not exact like Batch GD)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
plt.rcParams['figure.figsize'] = (12, 4)

## 1. Generate Data

In [ ]:
m = 1000
X = np.random.randn(m, 1)
y = 3 * X + 2 + np.random.randn(m, 1) * 0.5

plt.scatter(X, y, alpha=0.3, s=10, color='seagreen')
plt.xlabel('X')
plt.ylabel('y')
plt.title('Training Data  (y = 3x + 2 + noise)')
plt.show()

## 2. Helper Functions

In [ ]:
def predict(X, W, b):
    return X @ W + b

def mse_loss(y_pred, y_true):
    return np.mean((y_pred - y_true) ** 2)

def compute_gradients(X, y, W, b):
    n = len(y)
    error = predict(X, W, b) - y
    dW = (2 / n) * X.T @ error
    db = (2 / n) * np.sum(error)
    return dW, db

## 3. Mini-Batch GD — Implementation

Each epoch:
1. Shuffle the dataset
2. Split into batches of size B
3. For each batch: compute gradient → update weights
4. = **m/B updates per epoch**

In [ ]:
def mini_batch_gradient_descent(X, y, lr=0.05, epochs=100, batch_size=32):
    W = np.zeros((X.shape[1], 1))
    b = 0.0
    history = {'loss': [], 'W': [], 'b': []}
    n_batches = len(y) // batch_size

    for epoch in range(epochs):
        # Shuffle data each epoch
        indices = np.random.permutation(len(y))
        X_s = X[indices]
        y_s = y[indices]

        for start in range(0, len(y), batch_size):
            X_batch = X_s[start : start + batch_size]  # slice batch
            y_batch = y_s[start : start + batch_size]

            dW, db = compute_gradients(X_batch, y_batch, W, b)
            W -= lr * dW
            b -= lr * db

        loss = mse_loss(predict(X, W, b), y)
        history['loss'].append(loss)
        history['W'].append(W[0, 0])
        history['b'].append(b)

    return W, b, history

W_final, b_final, history = mini_batch_gradient_descent(X, y, lr=0.05, epochs=100, batch_size=32)

print(f"Learned  ->  W = {W_final[0,0]:.4f},  b = {b_final:.4f}")
print(f"True     ->  W = 3.0000,  b = 2.0000")
print(f"Final MSE Loss: {history['loss'][-1]:.4f}")
print(f"Updates per epoch: {m // 32} (batch_size=32)")

## 4. Visualize Results

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(history['loss'], color='seagreen', linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('MSE Loss')
axes[0].set_title('Loss Curve — Mini-Batch GD')
axes[0].grid(True, alpha=0.3)

axes[1].plot(history['W'], color='seagreen', linewidth=2)
axes[1].axhline(y=3, color='red', linestyle='--', label='True W=3')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('W')
axes[1].set_title('Weight W Convergence')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

axes[2].plot(history['b'], color='seagreen', linewidth=2)
axes[2].axhline(y=2, color='red', linestyle='--', label='True b=2')
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('b')
axes[2].set_title('Bias b Convergence')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.suptitle('Mini-Batch Gradient Descent  (batch_size=32)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Effect of Batch Size — The Spectrum

B=1 is pure SGD, B=m is Batch GD. Let's see the full spectrum.

In [ ]:
batch_configs = [
    (1,    'SGD (B=1)',          'red'),
    (8,    'Mini-Batch B=8',     'orange'),
    (32,   'Mini-Batch B=32',    'seagreen'),
    (128,  'Mini-Batch B=128',   'teal'),
    (256,  'Mini-Batch B=256',   'steelblue'),
    (1000, 'Batch GD (B=m)',     'purple'),
]

plt.figure(figsize=(13, 5))
for bs, label, color in batch_configs:
    _, _, hist = mini_batch_gradient_descent(X, y, lr=0.05, epochs=80, batch_size=bs)
    plt.plot(hist['loss'], label=label, color=color, linewidth=2 if bs in [32, 1000, 1] else 1.5)

plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.title('Mini-Batch GD: Effect of Batch Size  (B=1 → SGD,  B=m → Batch GD)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print("Observations:")
print("  B=1   (SGD)     -> Fastest start, most noisy, oscillates")
print("  B=32  (default) -> Good balance: fast + smooth enough")
print("  B=m   (Batch)   -> Smoothest curve, but slowest to start improving")

## 6. Side-by-Side: All Three Algorithms

In [ ]:
# Batch GD
W_b, b_b = np.zeros((1,1)), 0.0
loss_bgd = []
for _ in range(100):
    dW, db = compute_gradients(X, y, W_b, b_b)
    W_b -= 0.1 * dW; b_b -= 0.1 * db
    loss_bgd.append(mse_loss(predict(X, W_b, b_b), y))

# SGD
W_s, b_s = np.zeros((1,1)), 0.0
loss_sgd = []
for _ in range(100):
    for i in np.random.permutation(m):
        dW, db = compute_gradients(X[i:i+1], y[i:i+1], W_s, b_s)
        W_s -= 0.01 * dW; b_s -= 0.01 * db
    loss_sgd.append(mse_loss(predict(X, W_s, b_s), y))

# Mini-Batch
_, _, hist_mb = mini_batch_gradient_descent(X, y, lr=0.05, epochs=100, batch_size=32)

plt.figure(figsize=(13, 5))
plt.plot(loss_bgd,        label='Batch GD       (smooth, 1 update/epoch)',  color='steelblue', linewidth=2.5)
plt.plot(loss_sgd,        label='SGD            (noisy,  m updates/epoch)', color='darkorange', linewidth=1.5, alpha=0.8)
plt.plot(hist_mb['loss'], label='Mini-Batch GD  (balanced, B=32)',          color='seagreen',  linewidth=2.5)
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.title('All Three Gradient Descent Variants — Comparison')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 7. Full Comparison Table

| Property | Batch GD | SGD | Mini-Batch GD |
|----------|:--------:|:---:|:-------------:|
| Samples per update | All (m) | 1 | B (32–256) |
| Updates per epoch | 1 | m | m/B |
| Gradient accuracy | Exact | Noisy (high var) | Approximate |
| Loss curve | Smooth | Very noisy | Slightly noisy |
| Memory per step | High (all data) | Low (1 sample) | Medium (B samples) |
| GPU efficiency | Low | Very low | **High** |
| Convergence speed | Slow | Fast early | **Fast + stable** |
| Can escape local minima | No | Yes | Somewhat |
| Online learning | No | Yes | No |
| Default in practice | Rarely | Sometimes | **YES** |

### Recommended Batch Sizes

| Batch Size | Behavior |
|-----------|----------|
| 32 | Most common default; good for small models |
| 64 | Common; slightly smoother gradients |
| 128 | Large models; needs more GPU RAM |
| 256+ | Very large models / distributed training |

> **Note:** In deep learning papers and libraries (PyTorch, TensorFlow), `SGD` usually means **Mini-Batch SGD**. Pure SGD (B=1) is rarely used.

---

**Next:** `02-Perceptron-Activation-Functions/`